In [ ]:
import jpype
import os
# jvm 강제 종료
if jpype.isJVMStarted():
  jpype.shutdownJVM()
  
# 환경 변수에 JAVA_HOME이라는 변수들 등록
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-21.0.10'

# 일반 문자열에서 \는 기능이 있음. \문자를 하려면 \\해야함
from konlpy.tag import Okt
okt = Okt()

In [ ]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('sample.csv')

# print(dataset)

In [ ]:
# 전처리 결측치가 있으면 삭제
dataset = dataset.dropna()

In [ ]:
# 형태소 분석
corpus = dataset['sentence'].tolist()

# 명사, 동사, 형용사를 골라내는 작업
clean_corpus = []

for text in corpus:
	# stem=True : 동사를 동사 원형으로 바꿈
	result = okt.pos(text, stem=True)
	words = []

	for word, pos in result:
		
		# 단어의 품사가 명사, 동사, 형용사이면 추출하여 리스트에 추가
		if pos in ['Noun', 'Verb', 'Adjective']:
			words.append(word)
			
	# 리스트에 있는 단어들을 문자열로 변환 => 명사, 동사, 형용사만 있는 문자로 변환
	new_text = " ".join(words)
	clean_corpus.append(new_text)

dataset['cleaned_sentence'] = clean_corpus

dataset2 = dataset[['id', 'sentence', 'label', 'cleaned_sentence']]

# print(dataset2)

In [ ]:
def text_processing(text):
  result = okt.pos(text, stem=True)
  words = []
  for word, pos in result:
    if pos in ['Noun', 'Verb', 'Adjective']:
      words.append(word)
  new_text = " ".join(words)
  return new_text

In [ ]:
# text_processing('오늘 날씨가 정말 화창해서 기분이 최고예요')

dataset2['cleaned_sentence2'] = dataset2['sentence'].apply(text_processing)

# dataset2

In [ ]:
# 백터화
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

# 독립, 종속 분리
X = vectorizer.fit_transform(dataset2['cleaned_sentence'])
y = dataset['label']

from sklearn.model_selection import train_test_split
# 훈련, 테스트 분리하여 학습 후의 정확도를 확인
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
# 테스트
text = '자격증 시험에 합격했습니다'
vector_text = vectorizer.transform([text])
res = model.predict(vector_text)
print(f'{text} : {'긍정' if res[0] else '부정'}')

# 시험에 떨어지다라는 데이터를 학습하지 못해서 결과가 부정확
text = '날씨가 화창한데 자격증 시험에 떨어졌습니다'
vector_text = vectorizer.transform([text])
res = model.predict(vector_text)
print(f'{text} : {'긍정' if res[0] else '부정'}')